In [26]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

In [27]:
from langgraph.checkpoint.memory import InMemorySaver # Its for store memory in ram 

In [28]:
# Load environment variables
load_dotenv()

True

In [29]:
llm = ChatGroq(model_name="deepseek-r1-distill-llama-70b", api_key=os.getenv("GROQ_API_KEY"))

In [30]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [31]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [32]:
def generate_explanation(state: JokeState):

    prompt = f'write an expalanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [34]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [35]:
config1 = {"configurable":{"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': '<think>\nAlright, so I need to come up with a pizza joke. Hmm, where do I start? Well, I know pizza is a popular topic, so there are probably a lot of common jokes out there. Let me think about the components of pizza and see if I can find something funny.\n\nOkay, pizza has a crust, sauce, cheese, and toppings. Maybe I can play with one of those. Cheese is a big part of pizza, so maybe something with cheese. I remember that mozzarella is the most common cheese used on pizza. So maybe a pun involving mozzarella?\n\nWait, mozzarella sounds a bit like "motors" or something. Hmm, not sure. Alternatively, maybe something about how cheese melts or stretches. Oh, I know, when you take a bite of pizza, the cheese stretches, right? So maybe something about that.\n\nWait, maybe a play on words. Like, why did the mozzarella go to the party? Because it wanted to have a gouda time! Wait, that\'s using gouda, which is another type of cheese. Maybe that\'s a stretch. Or 

In [36]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': '<think>\nAlright, so I need to come up with a pizza joke. Hmm, where do I start? Well, I know pizza is a popular topic, so there are probably a lot of common jokes out there. Let me think about the components of pizza and see if I can find something funny.\n\nOkay, pizza has a crust, sauce, cheese, and toppings. Maybe I can play with one of those. Cheese is a big part of pizza, so maybe something with cheese. I remember that mozzarella is the most common cheese used on pizza. So maybe a pun involving mozzarella?\n\nWait, mozzarella sounds a bit like "motors" or something. Hmm, not sure. Alternatively, maybe something about how cheese melts or stretches. Oh, I know, when you take a bite of pizza, the cheese stretches, right? So maybe something about that.\n\nWait, maybe a play on words. Like, why did the mozzarella go to the party? Because it wanted to have a gouda time! Wait, that\'s using gouda, which is another type of cheese. Maybe th

In [37]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '<think>\nAlright, so I need to come up with a pizza joke. Hmm, where do I start? Well, I know pizza is a popular topic, so there are probably a lot of common jokes out there. Let me think about the components of pizza and see if I can find something funny.\n\nOkay, pizza has a crust, sauce, cheese, and toppings. Maybe I can play with one of those. Cheese is a big part of pizza, so maybe something with cheese. I remember that mozzarella is the most common cheese used on pizza. So maybe a pun involving mozzarella?\n\nWait, mozzarella sounds a bit like "motors" or something. Hmm, not sure. Alternatively, maybe something about how cheese melts or stretches. Oh, I know, when you take a bite of pizza, the cheese stretches, right? So maybe something about that.\n\nWait, maybe a play on words. Like, why did the mozzarella go to the party? Because it wanted to have a gouda time! Wait, that\'s using gouda, which is another type of cheese. Maybe t

In [38]:
# We get 4 values according to workflow stages

In [39]:
config2 = {"configurable":{"thread_id":"2"}}
workflow.invoke({'topic':'pasta'},config=config2)

{'topic': 'pasta',
 'joke': '<think>\nOkay, so I need to come up with a joke about pasta. Let me think about how to approach this. Jokes often play on words, so maybe I can find a pun related to pasta. \n\nFirst, I should consider different types of pasta and their characteristics. There\'s spaghetti, which is long and stringy. Maybe I can use that. Also, pasta is often associated with sauce, so perhaps something with red sauce?\n\nWait, the user provided an example joke: "Why did the spaghetti refuse to get married? Because it was afraid of getting tangled up in a saucy relationship!" That\'s a good example. It uses "tangled" because spaghetti is long and stringy, and "saucy" as a pun on both the sauce and a romantic relationship.\n\nSo, to create another joke, I should think of other pasta types and their traits. Let\'s see, maybe fusilli? It\'s twisted, so that could be a good angle. Or maybe something with lasagna layers? Or perhaps ravioli, which is stuffed.\n\nAlternatively, I ca

In [43]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': '<think>\nOkay, so I need to come up with a joke about pasta. Let me think about how to approach this. Jokes often play on words, so maybe I can find a pun related to pasta. \n\nFirst, I should consider different types of pasta and their characteristics. There\'s spaghetti, which is long and stringy. Maybe I can use that. Also, pasta is often associated with sauce, so perhaps something with red sauce?\n\nWait, the user provided an example joke: "Why did the spaghetti refuse to get married? Because it was afraid of getting tangled up in a saucy relationship!" That\'s a good example. It uses "tangled" because spaghetti is long and stringy, and "saucy" as a pun on both the sauce and a romantic relationship.\n\nSo, to create another joke, I should think of other pasta types and their traits. Let\'s see, maybe fusilli? It\'s twisted, so that could be a good angle. Or maybe something with lasagna layers? Or perhaps ravioli, which is stuffed.\n\

In [41]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': '<think>\nOkay, so I need to come up with a joke about pasta. Let me think about how to approach this. Jokes often play on words, so maybe I can find a pun related to pasta. \n\nFirst, I should consider different types of pasta and their characteristics. There\'s spaghetti, which is long and stringy. Maybe I can use that. Also, pasta is often associated with sauce, so perhaps something with red sauce?\n\nWait, the user provided an example joke: "Why did the spaghetti refuse to get married? Because it was afraid of getting tangled up in a saucy relationship!" That\'s a good example. It uses "tangled" because spaghetti is long and stringy, and "saucy" as a pun on both the sauce and a romantic relationship.\n\nSo, to create another joke, I should think of other pasta types and their traits. Let\'s see, maybe fusilli? It\'s twisted, so that could be a good angle. Or maybe something with lasagna layers? Or perhaps ravioli, which is stuffed.\n

Time Travel

In [44]:
workflow.get_state({"configurable": {"thread_id":"1","checkpoint_id":"1f06df5f-3f6a-61c6-8000-d533d3fc5262"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06df5f-3f6a-61c6-8000-d533d3fc5262'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2025-07-31T10:05:57.592702+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06df5f-3f58-6eda-bfff-6dc4b0c38296'}}, tasks=(PregelTask(id='42d680cd-6b49-74ed-2bcd-d5fdb4d68a74', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': '<think>\nAlright, so I need to come up with a pizza joke. Hmm, where do I start? Well, I know pizza is a popular topic, so there are probably a lot of common jokes out there. Let me think about the components of pizza and see if I can find something funny.\n\nOkay, pizza has a crust, sauce, cheese, and toppings. Maybe I can play with one of those. Cheese is a big part of pizza, so maybe something with cheese. I remem

In [45]:
workflow.invoke(None,{"configurable":{"thread_id":"1","checkpoint_id":"1f06df5f-3f6a-61c6-8000-d533d3fc5262"}})

{'topic': 'pizza',
 'joke': '<think>\nOkay, so the user wants a pizza joke. Let me think about common pizza-related topics. Delivery guys, toppings, cheese, maybe something with the pizza box or delivery time. \n\nHmm, delivery guys are a good target because they\'re relatable. Maybe something about the pizza delivery guy being in a hurry because the pizza gets cold. But I want a twist. Maybe a reason why he\'s slow. \n\nWait, maybe play on words. "Dough" as in the pizza dough and "go" as in moving. That could work. So, "Why did the pizza delivery guy quit his job? Because he couldn’t dough it anymore!" \n\nDoes that make sense? Yeah, it\'s a pun on "do" and "dough." It\'s simple and cheesy, which is perfect for a joke. I think that\'s a solid one. Let me write that down.\n</think>\n\nWhy did the pizza delivery guy quit his job?  \nBecause he couldn’t dough it anymore!',
 'explanation': '\n\nWhy did the pizza delivery guy quit his job?  \nBecause he couldn’t dough it anymore!'}

In [46]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '<think>\nOkay, so the user wants a pizza joke. Let me think about common pizza-related topics. Delivery guys, toppings, cheese, maybe something with the pizza box or delivery time. \n\nHmm, delivery guys are a good target because they\'re relatable. Maybe something about the pizza delivery guy being in a hurry because the pizza gets cold. But I want a twist. Maybe a reason why he\'s slow. \n\nWait, maybe play on words. "Dough" as in the pizza dough and "go" as in moving. That could work. So, "Why did the pizza delivery guy quit his job? Because he couldn’t dough it anymore!" \n\nDoes that make sense? Yeah, it\'s a pun on "do" and "dough." It\'s simple and cheesy, which is perfect for a joke. I think that\'s a solid one. Let me write that down.\n</think>\n\nWhy did the pizza delivery guy quit his job?  \nBecause he couldn’t dough it anymore!', 'explanation': '\n\nWhy did the pizza delivery guy quit his job?  \nBecause he couldn’t dough i

Updating State

In [48]:
workflow.update_state(
    {
        "configurable": {
            "thread_id": "1",
            "checkpoint_id": "1f06df5f-3f6a-61c6-8000-d533d3fc5262",
            "checkpoint_ns": ""
        }
    },
    {'topic': 'samosa'}
)

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f06dfda-97eb-6fd5-8001-a4c4fa3d567c'}}

In [49]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06dfda-97eb-6fd5-8001-a4c4fa3d567c'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2025-07-31T11:01:08.629463+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06df5f-3f6a-61c6-8000-d533d3fc5262'}}, tasks=(PregelTask(id='82823897-5463-ddd1-8138-feca8f9b534a', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': '<think>\nOkay, so the user wants a pizza joke. Let me think about common pizza-related topics. Delivery guys, toppings, cheese, maybe something with the pizza box or delivery time. \n\nHmm, delivery guys are a good target because they\'re relatable. Maybe something about the pizza delivery guy being in a hurry because the pizza 

In [50]:
workflow.invoke(None, {"configurable":{"thread_id": "1", "checkpoint_id":"1f06dfda-97eb-6fd5-8001-a4c4fa3d567c"}})

{'topic': 'samosa',
 'joke': '<think>\nOkay, so I need to come up with a joke about samosas. Hmm, where do I start? Well, I know that samosas are these delicious fried or baked pastries with savory fillings, usually served as snacks. They\'re popular in a lot of countries, especially in South Asia and the Middle East.\n\nI remember that the user provided an example joke: "Why did the samosa go to the doctor? Because it was feeling a little \'crusty\'!" That\'s a play on words with "crusty" referring both to the pastry\'s texture and feeling unwell. So, the structure is a question setup leading to a pun in the punchline.\n\nI should think of other wordplays related to samosas. Let me brainstorm some aspects of samosas: they\'re triangular, crispy, have fillings like potatoes, peas, onions, sometimes meat. They can be served with chutneys. They\'re often eaten at parties or gatherings.\n\nMaybe I can play on the shape. "Triangular" could lead to something, but I\'m not sure. Or maybe the

In [51]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': '<think>\nOkay, so I need to come up with a joke about samosas. Hmm, where do I start? Well, I know that samosas are these delicious fried or baked pastries with savory fillings, usually served as snacks. They\'re popular in a lot of countries, especially in South Asia and the Middle East.\n\nI remember that the user provided an example joke: "Why did the samosa go to the doctor? Because it was feeling a little \'crusty\'!" That\'s a play on words with "crusty" referring both to the pastry\'s texture and feeling unwell. So, the structure is a question setup leading to a pun in the punchline.\n\nI should think of other wordplays related to samosas. Let me brainstorm some aspects of samosas: they\'re triangular, crispy, have fillings like potatoes, peas, onions, sometimes meat. They can be served with chutneys. They\'re often eaten at parties or gatherings.\n\nMaybe I can play on the shape. "Triangular" could lead to something, but I\'m n

Fault Tolerance